# FORGE-MA: Train LLMs to Investigate Misinformation via GRPO

This notebook trains a Qwen2.5-3B-Instruct model on the FORGE-MA adversarial misinformation forensics environment using GRPO.

**Runtime**: T4 GPU (free tier) â€” ~45 minutes

**Requirements**: HuggingFace account + FORGE-MA Space running at `https://YOUR-USERNAME-forge-ma.hf.space`

In [ ]:
!git clone https://huggingface.co/spaces/NeuralHU/forge-rl
import sys
sys.path.insert(0, "./forge-rl")
!pip install -q unsloth trl datasets transformers accelerate peft
!pip install -q torch-geometric stix2
print("Install complete.")


In [ ]:
# Cell 2 â€” Load model with Unsloth
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct",  # 3B fits free T4
    max_seq_length=1024,
    load_in_4bit=True,
    dtype=None,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)
print("Model loaded.")

In [ ]:
# Cell 3 — Environment reward function
# Uses ForgeEnv directly (local). No HTTP client. No ForgeAction.
# ACTIONS (canonical, from env/misinfo_env.py):
#   0=query_source  1=trace_origin  2=cross_reference  3=request_context
#   4=entity_link   5=temporal_audit  6=network_cluster  7=flag_manipulation
#   8=submit_verdict_real  9=submit_verdict_misinfo  10=submit_verdict_satire
#   11=submit_verdict_out_of_context  12=submit_verdict_fabricated

from env.misinfo_env import MisInfoForensicsEnv
from env.forge_env import ForgeEnv, ForgeEnvConfig
import numpy as np

TOOL_ACTIONS = [0, 5, 2]  # query_source, temporal_audit (5 not 3), cross_reference

def _parse_verdict(text: str) -> int:
    """Map model output text → correct action index."""
    t = text.lower()
    if any(w in t for w in ["fabricat", "fabricated"]):
        return 12   # submit_verdict_fabricated
    if any(w in t for w in ["misinfo", "false", "manipulat", "mislead"]):
        return 9    # submit_verdict_misinfo  (NOT 10 — 10 is satire)
    if any(w in t for w in ["satire", "parody", "joke", "humor"]):
        return 10   # submit_verdict_satire
    if any(w in t for w in ["out of context", "recontextual", "cropped"]):
        return 11   # submit_verdict_out_of_context
    if any(w in t for w in ["verified", "true", "legitimate", "accurate", "real"]):
        return 8    # submit_verdict_real
    return 9        # default: misinfo (not 10)

def reward_fn(prompts, completions, claim_texts=None, **kwargs):
    """
    GRPO reward function.
    Each completion is evaluated on the SAME claim that was in its prompt.
    Uses ForgeEnv locally — no HTTP.
    Returns List[float] clipped to (0.001, 0.999).
    """
    rewards = []
    for i, completion in enumerate(completions):
        comp_text = (
            completion[-1]["content"]
            if isinstance(completion, list)
            else str(completion)
        )
        # Get the claim that was in this prompt
        claim = (claim_texts[i] if claim_texts else None)
        try:
            cfg = ForgeEnvConfig(budget=6, seed=i)
            env = ForgeEnv(cfg)
            obs, info = env.reset()
            # If we have the specific claim, override env's random one
            if claim:
                env._claim_text = claim
            # Run investigation tools
            for act in TOOL_ACTIONS:
                if env._done:
                    break
                env.step(act)
            # Submit verdict from model output
            verdict_action = _parse_verdict(comp_text)
            _, reward, _, _, _ = env.step(verdict_action)
            rewards.append(float(np.clip(reward, 0.001, 0.999)))
        except Exception as e:
            print(f"[reward_fn] episode {i} error: {e}")
            rewards.append(0.001)
    return rewards

print("reward_fn ready.")


In [ ]:
# Cell 4 — Dataset
# Each row has a DIFFERENT claim so GRPO has variance to learn from.
# claim_texts is passed as extra column into reward_fn via **kwargs.
from datasets import Dataset
from env.misinfo_env import MisInfoForensicsEnv
import random

INVESTIGATION_PROMPT_TEMPLATE = """\
You are a misinformation forensics agent. You have investigated the following claim:

CLAIM: {claim}

Your investigation found:
- Source credibility signals from query_source
- Temporal consistency from temporal_audit
- Cross-reference matches from cross_reference

Based on this investigation, submit your verdict. Respond with exactly one of:
- "This is MISINFO because [reason]"
- "This is SATIRE because [reason]"
- "This is VERIFIED because [reason]"
- "This is FABRICATED because [reason]"
- "This is OUT OF CONTEXT because [reason]"

Your verdict:"""

# Generate diverse claims from the actual task generators
TASK_NAMES = [
    "fabricated_stats", "out_of_context", "coordinated_campaign",
    "satire_news", "verified_fact", "politifact_liar",
    "image_forensics", "sec_fraud",
]

def _get_claim_for_task(task_name: str, seed: int) -> str:
    try:
        env = MisInfoForensicsEnv(task_names=[task_name], difficulty=1)
        obs, info = env.reset(seed=seed)
        from env.claim_graph import ClaimGraph
        root = env.graph.root
        return getattr(root, "text", f"Unverified claim about {task_name}")
    except Exception:
        return f"Unverified claim #{seed} about {task_name}"

N = 200  # enough for meaningful GRPO, small enough for T4 time limit
random.seed(42)
claims, prompts = [], []
for i in range(N):
    task = TASK_NAMES[i % len(TASK_NAMES)]
    claim = _get_claim_for_task(task, seed=i)
    claims.append(claim)
    prompts.append([{"role": "user", "content":
                      INVESTIGATION_PROMPT_TEMPLATE.format(claim=claim)}])

dataset = Dataset.from_dict({"prompt": prompts, "claim_texts": claims})
print(f"Dataset: {len(dataset)} rows across {len(TASK_NAMES)} task types")
print(f"Sample claim: {claims[0][:100]}...")


In [ ]:
# Cell 5 — Baseline: heuristic agent (always predict misinfo=9)
# Records baseline BEFORE training. Uses same ForgeEnv, same actions.
import numpy as np

def evaluate_heuristic(n_episodes=20, default_action=9):
    """Heuristic: always submit one fixed verdict. Establishes floor."""
    rewards, correct = [], 0
    for i in range(n_episodes):
        try:
            from env.forge_env import ForgeEnv, ForgeEnvConfig
            cfg = ForgeEnvConfig(budget=6, seed=100 + i)
            env = ForgeEnv(cfg)
            env.reset()
            for act in TOOL_ACTIONS:
                if env._done: break
                env.step(act)
            _, reward, _, _, info = env.step(default_action)
            rewards.append(float(reward))
            if reward > 0.5:
                correct += 1
        except Exception as e:
            print(f"Eval error ep {i}: {e}")
            rewards.append(0.001)
    return float(np.mean(rewards)), correct / n_episodes

baseline_reward, baseline_acc = evaluate_heuristic()
print(f"HEURISTIC BASELINE — reward: {baseline_reward:.4f} | acc: {baseline_acc:.2%}")
print("(This is what random misinfo-always prediction achieves.)")


In [ ]:
# Cell 6 — Train with GRPO
from trl import GRPOConfig, GRPOTrainer

config = GRPOConfig(
    output_dir="./forge-grpo",
    max_steps=150,              # NOT num_train_epochs — avoids timeout on T4
    num_generations=4,          # completions per prompt per GRPO step
    max_completion_length=128,  # verdicts are short — 256 wastes tokens
    per_device_train_batch_size=1,   # 1 not 2 — 4 generations × 3B model = tight
    gradient_accumulation_steps=8,
    learning_rate=5e-6,
    save_steps=50,
    logging_steps=5,
    report_to=[],               # NOT "none" (string) — empty list is correct
    warmup_ratio=0.1,
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=reward_fn,
    args=config,
    train_dataset=dataset,
    processing_class=tokenizer,   # 'tokenizer' arg deprecated in TRL>=0.9
)

print("Starting GRPO training (max 150 steps)...")
trainer.train()
print("Training complete.")


In [ ]:
# Cell 7 — Post-training evaluation + plots
# evaluate_trained() uses the ACTUAL model to generate verdicts.
import matplotlib.pyplot as plt, matplotlib, os, numpy as np
matplotlib.use('Agg')
os.makedirs("results", exist_ok=True)

def evaluate_trained(n_episodes=20):
    """Run the trained model on new claims and score with ForgeEnv."""
    rewards, correct = [], 0
    from env.forge_env import ForgeEnv, ForgeEnvConfig
    FastLanguageModel.for_inference(model)   # re-enable Unsloth inference mode

    for i in range(n_episodes):
        try:
            # Generate a fresh claim
            task = TASK_NAMES[i % len(TASK_NAMES)]
            claim = _get_claim_for_task(task, seed=200 + i)
            prompt_text = INVESTIGATION_PROMPT_TEMPLATE.format(claim=claim)
            inputs = tokenizer([{"role":"user","content":prompt_text}],
                                return_tensors="pt").to("cuda")
            outputs = model.generate(**inputs, max_new_tokens=80,
                                     temperature=0.3, do_sample=True)
            response = tokenizer.decode(outputs[0], skip_special_tokens=True)
            verdict_action = _parse_verdict(response)

            cfg = ForgeEnvConfig(budget=6, seed=200 + i)
            env = ForgeEnv(cfg)
            env.reset()
            env._claim_text = claim
            for act in TOOL_ACTIONS:
                if env._done: break
                env.step(act)
            _, reward, _, _, _ = env.step(verdict_action)
            rewards.append(float(reward))
            if reward > 0.5:
                correct += 1
        except Exception as e:
            print(f"Trained eval error ep {i}: {e}")
            rewards.append(0.001)
    return float(np.mean(rewards)), correct / n_episodes

trained_reward, trained_acc = evaluate_trained()
print(f"BASELINE  — reward: {baseline_reward:.4f} | acc: {baseline_acc:.2%}")
print(f"TRAINED   — reward: {trained_reward:.4f} | acc: {trained_acc:.2%}")
print(f"DELTA     — {trained_reward-baseline_reward:+.4f} reward | {trained_acc-baseline_acc:+.2%} acc")

# Reward curve — TRL logs 'reward/mean' (no 's' prefix)
log_history = trainer.state.log_history
steps   = [l["step"]         for l in log_history if "reward/mean" in l]
rewards = [l["reward/mean"]  for l in log_history if "reward/mean" in l]

fig, ax = plt.subplots(figsize=(10, 5))
if steps:
    ax.plot(steps, rewards, 'b-', linewidth=2.5, label='Training reward')
ax.axhline(y=baseline_reward, color='r', linestyle='--',
           linewidth=2, label=f'Heuristic baseline ({baseline_reward:.3f})')
ax.set_xlabel("Training Step", fontsize=13)
ax.set_ylabel("Mean Episode Reward", fontsize=13)
ax.set_title("FORGE-MA: LLM Agent via GRPO", fontsize=14, fontweight='bold')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1.0)
plt.tight_layout()
plt.savefig("results/reward_curve.png", dpi=150, bbox_inches='tight')
plt.close()

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(['Heuristic Baseline','GRPO Trained'],
              [baseline_reward, trained_reward],
              color=['#e74c3c','#2ecc71'], edgecolor='black', width=0.5)
ax.set_ylabel("Mean Episode Reward", fontsize=13)
ax.set_title("Before vs After GRPO\n(FORGE-MA)", fontsize=13)
ax.set_ylim(0, 1.0)
for bar, val in zip(bars, [baseline_reward, trained_reward]):
    ax.text(bar.get_x()+bar.get_width()/2., bar.get_height()+0.02,
            f'{val:.3f}', ha='center', va='bottom', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("results/before_after.png", dpi=150, bbox_inches='tight')
plt.close()
print("Saved: results/reward_curve.png + results/before_after.png")
print("COMMIT BOTH PNG FILES TO REPO BEFORE SUBMITTING")
